# Pipeline do modelo - casos de teste

Fluxo: `cases.csv` -> split treino/teste (por artigo) -> pre-processamento -> rotulacao fraca (treino, MeSH) -> features por token + regressao logistica -> predicao no teste -> pos-processamento.

Estrutura de pastas esperada:

```
project02/
  data/
    raw/sample/cases.csv
    external/MeSH/       mesh_anatomy.csv, mesh_diseases.csv, mesh_*.csv
  models/
    pre_processamento/   tokenizer.py, split_cases.py
    classificar_nos/     step2_weak_labels.py, features.py, ...
  notebook/              pipeline_modelo.ipynb   <- este arquivo
```

Rode o notebook com o diretorio de trabalho em `project02/notebook`.

## 1. Caminhos e imports

In [1]:
# Recarrega automaticamente os .py de project02/ quando voce os edita (evita ficar com versao antiga em memoria)
%load_ext autoreload
%autoreload 2

import sys
import json
from pathlib import Path

import pandas as pd

# project02/notebook -> project02 -> pasta que CONTEM project02 (necessaria para `import project02...`)
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_DIR = NOTEBOOK_DIR.parent
ROOT = PROJECT_DIR.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CASES_CSV = PROJECT_DIR / 'data' / 'raw' / 'sample' / 'cases.csv'
PRE_DIR = PROJECT_DIR / 'models' / 'pre_processamento'
NOS_DIR = PROJECT_DIR / 'models' / 'classificar_nos'
SPLITS_DIR = PROJECT_DIR / 'data' / 'processed' / 'splits'
OUT_DIR = PROJECT_DIR / 'data' / 'processed'

# confere se tudo esta no lugar
esperados = [
    CASES_CSV,
    PRE_DIR / 'tokenizer.py',
    PRE_DIR / 'split_cases.py',
    NOS_DIR / 'step2_weak_labels.py',
    NOS_DIR / 'features.py',
]
for p in esperados:
    print('OK     ' if p.exists() else 'FALTA  ', p.relative_to(ROOT))

# confere se o tokenizer.py e a versao nova (com os tokens com posicao, usados na rotulacao fraca)
if 'PREPROC_ID' not in (PRE_DIR / 'tokenizer.py').read_text(encoding='utf-8'):
    print('\nATENCAO: tokenizer.py esta desatualizado (sem PREPROC_ID). Substitua pelo arquivo novo.')

OK      project02\data\raw\sample\cases.csv
OK      project02\models\pre_processamento\tokenizer.py
OK      project02\models\pre_processamento\split_cases.py
OK      project02\models\classificar_nos\step2_weak_labels.py
OK      project02\models\classificar_nos\features.py


## 2. Parametros

Tudo definido aqui, sem arquivo de configuracao externo.

In [2]:
TEST_SIZE = 0.2          # fracao dos artigos que vai para o teste
SEED = 42                # para reproduzir o mesmo split
GROUP_COL = 'article_id' # casos do mesmo artigo ficam juntos (evita vazamento)
TEXT_COL = 'case_text'   # coluna com o texto do caso; None = detecta a coluna de texto mais longa

# Rotulacao fraca (passo 2)
MESH_DIR = PROJECT_DIR / 'data' / 'external' / 'MeSH'                          # mesh_*.csv
STOPLIST = PROJECT_DIR / 'models' / 'classificar_nos' / 'stoplist.txt'         # opcional: termos a ignorar, um por linha
WEAK_DIR = PROJECT_DIR / 'data' / 'processed' / 'weak_labels'                  # saida

# Modelo (passo 3)
DICT_DROPOUT = 0.5       # chance de esconder cada termo do dicionario no treino (0 = nunca, 1 = sempre)
WINDOW = 2               # vizinhos de cada lado usados como contexto
N_FOLDS = 5              # validacao cruzada por artigo, dentro do treino
MODEL_DIR = PROJECT_DIR / 'data' / 'processed' / 'model'                       # saida

## 3. Split treino / teste (agrupado por artigo)

In [3]:
from project02.models.pre_processamento.split_cases import split_cases

train, test = split_cases(
    cases_csv=CASES_CSV,
    out_dir=SPLITS_DIR,
    test_size=TEST_SIZE,
    seed=SEED,
    group_col=GROUP_COL,
)

print(f'treino: {len(train)} casos / {train[GROUP_COL].nunique()} artigos')
print(f'teste : {len(test)} casos / {test[GROUP_COL].nunique()} artigos')
print('salvo em', SPLITS_DIR.relative_to(ROOT))
print('colunas:', list(test.columns))
test.head(3)

treino: 45 casos / 40 artigos
teste : 11 casos / 10 artigos
salvo em project02\data\processed\splits
colunas: ['article_id', 'age', 'case_id', 'case_text', 'gender']


,article_id,age,case_id,case_text,gender
0,PMC5137649,44.0,PMC5137649_01,A 44-year-old woman presented with a 3-day his...,Female
8,PMC5833205,0.0,PMC5833205_01,A previously healthy 20-month-old boy presente...,Male
13,PMC9529523,41.0,PMC9529523_01,A 41-year-old African American female with an ...,Female


## 4. Pre-processamento do teste

Duas versoes de tokens: `tokens_lower` (minusculas, sem pontuacao) e `tokens_stem` (com radical).

In [4]:
from project02.models.pre_processamento.tokenizer import preprocess

if TEXT_COL is None:
    # is_string_dtype cobre tanto 'object' quanto o tipo 'str' das versoes novas do pandas
    ignorar = {'case_id', GROUP_COL}
    candidatas = [c for c in test.columns if pd.api.types.is_string_dtype(test[c]) and c not in ignorar]
    TEXT_COL = max(candidatas, key=lambda c: test[c].astype(str).str.len().mean())
print('coluna de texto:', TEXT_COL)

test = test.copy()
textos = test[TEXT_COL].fillna('').astype(str)
test['tokens_lower'] = textos.apply(lambda t: preprocess(t, use_stem=False))
test['tokens_stem'] = textos.apply(lambda t: preprocess(t, use_stem=True))

test[[TEXT_COL, 'tokens_lower', 'tokens_stem']].head(2)

coluna de texto: case_text


,case_text,tokens_lower,tokens_stem
0,A 44-year-old woman presented with a 3-day his...,"[a, 44-year-old, woman, presented, with, a, 3-...","[a, 44-year-old, woman, present, with, a, 3-da..."
8,A previously healthy 20-month-old boy presente...,"[a, previously, healthy, 20-month-old, boy, pr...","[a, previous, healthi, 20-month-old, boy, pres..."


## 5. Rotulacao fraca (treino)

Usa os dicionarios MeSH (`data/external/MeSH/mesh_*.csv`) para marcar, em cada caso de **treino**, os trechos
de Disease, Anatomy, Symptom, Medication etc. no formato BIO (match do termo mais longo). `Patient`
(idade e genero) sai de regras. O teste nao e rotulado aqui: ele e anotado a mao para medir o modelo.

Se algum `mesh_*.csv` nao existir, o tipo correspondente fica sem rotulos (o aviso aparece abaixo).

In [5]:
from project02.models.classificar_nos.step2_weak_labels import (
    build_dictionary, label_dataframe, save_weak_labels, summarize,
)

# 1) dicionario MeSH normalizado do mesmo jeito que o texto (minusculas, sem pontuacao, radical)
dic = build_dictionary(MESH_DIR, stoplist=STOPLIST)

# 2) tokeniza os casos de TREINO (com posicao no texto) e rotula pelo dicionario
records, metas, n_tokens = label_dataframe(
    train, dic, text_col=TEXT_COL, id_col='case_id', group_col=GROUP_COL,
)

# 3) salva weak_labels.jsonl e dictionary.json
weak_path = save_weak_labels(records, dic, WEAK_DIR)
print('salvo em', weak_path.relative_to(ROOT))

summarize(records, metas, n_tokens)

Dicionarios carregados: Anatomy, Disease, Exam, Medication, Result, Symptom, Procedure
Dicionario: 14165 termos normalizados (824 ambiguos, 1265 IGNORE)
salvo em project02\data\processed\weak_labels\weak_labels.jsonl

Casos: 45 | tokens: 26801 | tokens em span/IGNORE: 2349 (8.8%)
Spans por rotulo principal: {'Anatomy': 530, 'Patient': 331, 'Disease': 297, 'Exam': 234, 'IGNORE': 171, 'Medication': 145, 'Symptom': 141, 'Result': 69}
Spans ambiguos (candidates): {'Disease|Result': 46, 'Symptom|Disease': 31, 'Symptom|Result': 4, 'Symptom|Exam': 2, 'Symptom|Disease|Result': 1}
Patient/genero vs CSV: ok=41 divergente=1 nao_extraido=0
Patient/idade  vs CSV: ok=37 divergente=1 nao_extraido=2


In [12]:
# Revisao: quais termos mais geram span em cada rotulo? (ajuda a achar ruido para a stoplist)
todos = pd.DataFrame([dict(s, case_id=r['case_id']) for r in records for s in r['spans']])

if len(todos):
    topo = (todos.assign(termo=todos['text'].str.lower())
                 .groupby('label')['termo'].value_counts()
                 .groupby(level=0).head(8))
    display(topo.to_frame('n'))
else:
    print('Nenhum span encontrado: confira MESH_DIR e o texto dos casos.')

# Exemplo de um caso
r0 = records[0]
print(r0['case_id'], r0['patient'])
pd.DataFrame(r0['spans'])[['text', 'label', 'candidates', 'source']]

n
label   termo          
Anatomy blood        48
        abdomen      20
        tissue       13
        colon        12
        lung         11
...                  ..
Symptom chest pain   10
        edema         6
        diarrhea      5
        nausea        5
        weight loss   5

[64 rows x 1 columns]

PMC9387390_01 {'age_years': 57.0, 'gender': 'male', 'gender_source': 'head', 'has_mention': True}


,text,label,candidates,source
0,man,Patient,[Patient],rule
1,diarrhea,Symptom,[Symptom],dict
2,tomography,Exam,[Exam],dict
3,edema,Symptom,[Symptom],dict
4,barium enema,Exam,[Exam],dict
5,abdomen,Anatomy,[Anatomy],dict
6,pancreas,Anatomy,[Anatomy],dict
7,duodenum,Anatomy,[Anatomy],dict
8,patient,Patient,[Patient],rule
9,biopsy,Exam,[Exam],dict


## 6. Modelo: features por token + regressao logistica

Cada token vira um dicionario de features (forma da palavra, vizinhos numa janela, palavras de contexto e o
tipo dos termos do MeSH que aparecem ao redor) -> `DictVectorizer` -> `LogisticRegression` multinomial. Sem embeddings.

Dois cuidados para o modelo nao virar apenas uma copia do dicionario (ver `features.py`):

- **Dropout do dicionario:** no treino, cada termo do MeSH e escondido com probabilidade `DICT_DROPOUT`, entao o
  modelo tambem aprende pela forma da palavra e pelo contexto.
- **Termos ambiguos** (ex.: Disease|Result) ficam fora do treino, porque o rotulo fraco deles vem de uma ordem de
  prioridade, nao do contexto. Na predicao o contexto escolhe o tipo entre os candidatos do dicionario.

**Atencao:** a validacao abaixo compara com os rotulos *fracos*, que nascem do proprio dicionario. Serve para comparar
configuracoes (janela, dropout, C), nao como medida final. A medida final e no teste anotado a mao.

In [6]:
import numpy as np
from sklearn.metrics import classification_report
from sklearn.model_selection import GroupKFold

from project02.models.classificar_nos.features import build_dataset, make_model

# Tres visoes do MESMO conjunto de tokens (mesma ordem): dicionario escondido em parte dos termos (treino),
# dicionario visivel (como na predicao) e dicionario todo escondido (mede se o modelo generaliza sem ele).
kw = dict(text_col=TEXT_COL, id_col='case_id', window=WINDOW)
X_tr, y, ids = build_dataset(train, records, dict_dropout=DICT_DROPOUT, seed=SEED, **kw)
X_com, _, _ = build_dataset(train, records, dict_dropout=0.0, **kw)
X_sem, _, _ = build_dataset(train, records, dict_dropout=1.0, **kw)
y = np.array(y)

artigo = dict(zip(train['case_id'].astype(str), train[GROUP_COL]))
grupos = np.array([artigo[cid] for cid, _ in ids])
print(f'tokens de treino: {len(y)} | artigos: {len(set(grupos))}')

# Validacao cruzada por artigo: casos do mesmo artigo nunca ficam em treino e validacao ao mesmo tempo
n_splits = min(N_FOLDS, len(set(grupos)))
pred_com = np.empty(len(y), dtype=object)
pred_sem = np.empty(len(y), dtype=object)
for tr_idx, va_idx in GroupKFold(n_splits=n_splits).split(np.zeros(len(y)), y, grupos):
    m = make_model().fit([X_tr[k] for k in tr_idx], y[tr_idx])
    pred_com[va_idx] = m.predict([X_com[k] for k in va_idx])
    pred_sem[va_idx] = m.predict([X_sem[k] for k in va_idx])

print('=== COM dicionario (como na predicao) ===')
print(classification_report(y, pred_com, zero_division=0))
print('=== SEM dicionario (generalizacao: so forma da palavra e contexto) ===')
print(classification_report(y, pred_sem, zero_division=0))

tokens de treino: 26483 | artigos: 40
=== COM dicionario (como na predicao) ===
              precision    recall  f1-score   support

   B-Anatomy       0.87      1.00      0.93       530
   B-Disease       0.79      0.99      0.88       251
      B-Exam       0.87      1.00      0.93       234
B-Medication       0.78      0.97      0.86       145
   B-Patient       0.98      0.97      0.98       331
    B-Result       0.94      0.87      0.90        69
   B-Symptom       0.96      0.98      0.97       103
   I-Anatomy       0.83      0.97      0.89       100
   I-Disease       0.82      0.94      0.88        86
      I-Exam       0.81      0.98      0.89       126
I-Medication       0.80      0.31      0.44        13
   I-Patient       1.00      0.56      0.71         9
    I-Result       1.00      0.25      0.40         4
   I-Symptom       1.00      0.93      0.97        30
           O       1.00      0.99      0.99     24452

    accuracy                           0.99     26483


In [7]:
import joblib

# Modelo final: treina com todo o treino
model = make_model().fit(X_tr, y)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_DIR / 'model.joblib')
print('modelo salvo em', (MODEL_DIR / 'model.joblib').relative_to(ROOT))

# O que o modelo aprendeu: as features de maior peso em cada classe
vec, clf = model.steps[0][1], model.steps[1][1]
nomes = np.array(vec.get_feature_names_out())
for k, classe in enumerate(clf.classes_):
    if classe == 'O':
        continue
    top = nomes[np.argsort(clf.coef_[k])[-8:][::-1]]
    print(f'{classe:13}', ', '.join(top))

modelo salvo em project02\data\processed\model\model.joblib
B-Anatomy     dict:Anatomy, suf2=um, dict_first, -2:dict:Anatomy, -2:norm=the, -1:suf3=ght, pre3=lym, norm=neutrophil
B-Disease     dict:Disease, norm=infect, dict_first, lower=infection, suf2=al, pre3=pne, suf2=ia, suf2=is
B-Exam        dict:Exam, suf4=aphy, dict_first, pre3=bio, norm=dilat, pre3=dil, suf3=phy, norm=biopsi
B-Medication  dict:Medication, suf2=in, suf2=ne, dict_first, ctx:mg, suf3=cin, suf3=ine, suf3=rin
B-Patient     norm=patient, pre3=pat, suf4=male, suf3=ale, lower=patient, prev+cur=the|patient, suf2=le, suf4=ient
B-Result      dict:Result, norm=fistula, pre3=fis, pre3=hem, norm=ulcer, pre3=ulc, dict_first, pre3=inf
B-Symptom     dict:Symptom, dict_first, ctx:pain, pre3=fev, norm=fever, +1:suf3=ain, +1:norm=pain, suf2=ea
I-Anatomy     -1:dict:Anatomy, dict:Anatomy, dict_inside, norm=extrem, pre3=ext, suf2=ve, -1:suf3=ary, -1:suf3=nal
I-Disease     -1:dict:Disease, dict:Disease, dict_inside, ctx:diagnos, prev

## 7. Predicao no teste e pos-processamento

Para cada caso de teste: casa o texto com o dicionario (so para obter os candidatos), prediz as etiquetas BIO,
corrige BIO invalido (um `I-X` sem `B-X` antes vira `B-X`) e converte em entidades. Nos termos ambiguos o
contexto escolhe o tipo entre os candidatos, que tambem ficam guardados em `candidates`.

In [8]:
from project02.models.pre_processamento.tokenizer import process_case
from project02.models.classificar_nos.step2_weak_labels import label_case
from project02.models.classificar_nos.features import predict_case, fix_bio, bio_to_entities

entidades = []
for row in test.to_dict('records'):
    case = process_case(str(row['case_id']), str(row[GROUP_COL]), str(row[TEXT_COL]))
    record = label_case(case, dic)                        # aqui so interessa o casamento com o dicionario
    tags, cands = predict_case(model, case, record, window=WINDOW)
    tags = fix_bio(tags)                                  # pos-processamento: I- sem B- vira B-
    entidades.append(bio_to_entities(case, tags, cands))

test['entities'] = entidades
print('casos processados:', len(test))
print('casos com alguma entidade:', int((test['entities'].str.len() > 0).sum()))
print('entidades por tipo:', pd.Series([e['label'] for ents in entidades for e in ents]).value_counts().to_dict())

# Exemplo: primeiro caso do teste
ex = entidades[0]
pd.DataFrame(ex)[['text', 'label', 'candidates']] if ex else 'nenhuma entidade no primeiro caso'

casos processados: 11
casos com alguma entidade: 11
entidades por tipo: {'Anatomy': 111, 'Patient': 109, 'Disease': 93, 'Exam': 75, 'Medication': 30, 'Symptom': 29, 'Result': 10}


,text,label,candidates
0,woman,Patient,[]
1,abdominal pain,Symptom,[Symptom]
2,nausea,Symptom,[Symptom]
3,constipation,Symptom,[Symptom]
4,physical examination,Exam,[Exam]
5,tomography,Exam,[Exam]
6,stomach,Anatomy,[Anatomy]
7,pancreas,Anatomy,[Anatomy]
8,pancreatic,Disease,[Disease]
9,cyst,Disease,"[Disease, Result]"


## 8. Salvar predicoes

In [9]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

saida = test.copy()
for col in ['tokens_lower', 'tokens_stem', 'entities']:
    saida[col] = saida[col].apply(json.dumps)

destino = OUT_DIR / 'test_predictions.csv'
saida.to_csv(destino, index=False)
print('salvo em', destino.relative_to(ROOT))

salvo em project02\data\processed\test_predictions.csv
